# ID Quality Analysis — Uber NCR Ride Bookings

**Questions from:** `process/active_questions.md` → ID Quality

1. Why do 2,457 rows share non-unique Booking IDs with different dates, customers, and statuses?
2. Do duplicated Customer IDs (1,212) represent repeat customers, or malformed rows?

---

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../../..'))

import pandas as pd
import numpy as np
from pathlib import Path
from utils.analysis_log import AnalysisLog

OUTPUT_DIR = Path('../notebook_logs')
OUTPUT_DIR.mkdir(exist_ok=True)

log = AnalysisLog()
log.add(
    section_id='01_setup',
    title='Setup',
    cell_type='setup',
    purpose='Load libraries and initialize AnalysisLog for ID quality investigation',
    data={'libraries': ['pandas', 'numpy']},
)

## 1. Load Data

In [2]:
DATA_PATH = Path('..') / 'data' / 'ncr_ride_bookings.csv'
df = pd.read_csv(DATA_PATH)

log.add(
    section_id='02_load_data',
    title='Load Data',
    cell_type='setup',
    purpose='Load the dataset for ID quality analysis',
    data={'rows': df.shape[0], 'columns': df.shape[1]},
)
df.shape

(150000, 21)

## 2. Booking ID — Top-Level Counts

In [3]:
total_rows = len(df)
unique_booking_ids = df['Booking ID'].nunique()
# rows involved in any duplication (keep=False marks all copies)
dup_booking_mask = df['Booking ID'].duplicated(keep=False)
rows_in_dup_booking = int(dup_booking_mask.sum())
# unique IDs that appear more than once
booking_id_counts = df['Booking ID'].value_counts()
dup_booking_ids_count = int((booking_id_counts > 1).sum())

booking_summary = {
    'total_rows': total_rows,
    'unique_booking_ids': unique_booking_ids,
    'booking_ids_appearing_more_than_once': dup_booking_ids_count,
    'rows_involved_in_duplicate_booking_ids': rows_in_dup_booking,
    'pct_rows_affected': round(rows_in_dup_booking / total_rows * 100, 2),
}

log.add(
    section_id='03_booking_id_counts',
    title='Booking ID Duplicate Counts',
    cell_type='analysis',
    purpose='Quantify how many Booking IDs appear more than once and how many rows are affected',
    data=booking_summary,
)
booking_summary

{'total_rows': 150000,
 'unique_booking_ids': 148767,
 'booking_ids_appearing_more_than_once': 1224,
 'rows_involved_in_duplicate_booking_ids': 2457,
 'pct_rows_affected': 1.64}

## 3. Booking ID — Occurrence Frequency Distribution

If duplicated IDs always appear exactly twice, it suggests ID reuse (each booking ID used for 2 rides). If some IDs appear 3+ times it suggests a different pattern.

In [4]:
booking_freq = booking_id_counts.value_counts().sort_index()
booking_freq_dict = {f'appears_{k}_times': int(v) for k, v in booking_freq.items()}

log.add(
    section_id='04_booking_id_frequency_dist',
    title='Booking ID Occurrence Frequency Distribution',
    cell_type='analysis',
    purpose='Check whether duplicated Booking IDs appear exactly twice or more, to distinguish ID reuse from a generation artifact',
    data=booking_freq_dict,
)
booking_freq

count
1    147543
2      1215
3         9
Name: count, dtype: int64

## 4. Booking ID Duplicates — Are Rows Identical?

If the duplicate rows are identical on all fields, the duplication is a simple row-level copy.
If fields differ (date, customer, status), the ID is reused for different rides.

In [5]:
dup_booking_rows = df[dup_booking_mask].copy()

# Identical rows: all 21 columns the same
exact_dup_mask = df.duplicated(keep=False)
exact_dup_count = int(exact_dup_mask.sum())

# Among the booking-ID-duplicated rows, how many are also full-row duplicates?
both_mask = dup_booking_mask & exact_dup_mask
booking_dup_that_are_exact = int(both_mask.sum())

identical_rows_result = {
    'exact_duplicate_rows_in_full_dataset': exact_dup_count,
    'booking_id_dup_rows_that_are_also_exact_row_duplicates': booking_dup_that_are_exact,
    'booking_id_dup_rows_with_different_content': rows_in_dup_booking - booking_dup_that_are_exact,
}

log.add(
    section_id='05_identical_rows_check',
    title='Booking ID Duplicates — Identical vs Different Rows',
    cell_type='analysis',
    purpose='Determine whether Booking ID duplicates are row-level copies or ID reuse on genuinely different rides',
    data=identical_rows_result,
)
identical_rows_result

{'exact_duplicate_rows_in_full_dataset': 0,
 'booking_id_dup_rows_that_are_also_exact_row_duplicates': 0,
 'booking_id_dup_rows_with_different_content': 2457}

## 5. Booking ID Duplicates — Field Variation

For each duplicated Booking ID pair/group: across the fields Date, Time, Customer ID, Booking Status, Vehicle Type — how often do values differ?

In [6]:
check_cols = ['Date', 'Time', 'Customer ID', 'Booking Status', 'Vehicle Type']

# For each duplicated Booking ID group, check nunique per column
grouped = dup_booking_rows.groupby('Booking ID')[check_cols].nunique()

# A column varies within a group if nunique > 1
varies = (grouped > 1).sum()  # how many groups have variation in each column
total_groups = len(grouped)

field_variation = {
    'total_duplicated_booking_id_groups': total_groups,
    'groups_where_field_varies': {col: int(varies[col]) for col in check_cols},
    'pct_groups_where_field_varies': {col: round(int(varies[col]) / total_groups * 100, 1) for col in check_cols},
}

log.add(
    section_id='06_booking_dup_field_variation',
    title='Booking ID Duplicates — Field Variation',
    cell_type='analysis',
    purpose='Measure how often key fields differ across rows sharing the same Booking ID to characterize the nature of the duplication',
    data=field_variation,
)
field_variation

{'total_duplicated_booking_id_groups': 1224,
 'groups_where_field_varies': {'Date': 1219,
  'Time': 1224,
  'Customer ID': 1224,
  'Booking Status': 731,
  'Vehicle Type': 1020},
 'pct_groups_where_field_varies': {'Date': 99.6,
  'Time': 100.0,
  'Customer ID': 100.0,
  'Booking Status': 59.7,
  'Vehicle Type': 83.3}}

## 6. Booking ID Duplicates — Status Pair Combinations

What booking status combinations appear when the same ID is reused? (e.g., Completed + No Driver Found suggests the ID was recycled from a failed attempt.)

In [7]:
# For groups of size 2 exactly, build sorted status pair strings
groups_size2 = grouped[grouped.index.isin(
    booking_id_counts[booking_id_counts == 2].index
)]

status_pairs = (
    dup_booking_rows[dup_booking_rows['Booking ID'].isin(groups_size2.index)]
    .groupby('Booking ID')['Booking Status']
    .apply(lambda x: ' | '.join(sorted(x.values)))
    .value_counts()
)

log.add(
    section_id='07_status_pair_combinations',
    title='Booking ID Duplicates — Status Pair Combinations',
    cell_type='analysis',
    purpose='Identify what booking status combinations occur when the same ID appears twice, to understand if the duplication pattern is a retry/reuse artifact',
    data=status_pairs.to_dict(),
)
status_pairs

Booking Status
Completed | Completed                            433
Cancelled by Driver | Completed                  290
Cancelled by Customer | Completed                110
Completed | No Driver Found                      106
Completed | Incomplete                            93
Cancelled by Driver | No Driver Found             41
Cancelled by Driver | Cancelled by Driver         37
Cancelled by Customer | Cancelled by Driver       32
Incomplete | No Driver Found                      16
Cancelled by Customer | Incomplete                15
Cancelled by Driver | Incomplete                  15
Cancelled by Customer | No Driver Found           11
Incomplete | Incomplete                            6
No Driver Found | No Driver Found                  6
Cancelled by Customer | Cancelled by Customer      4
Name: count, dtype: int64

## 7. Booking ID Duplicates — Date Gap Between Pairs

If duplicate IDs represent retried bookings, the two rows should be close in date.
If they are spread across months, it's more likely a generation artifact (random ID collision).

In [8]:
size2_ids = booking_id_counts[booking_id_counts == 2].index

pairs_df = (
    dup_booking_rows[dup_booking_rows['Booking ID'].isin(size2_ids)]
    .assign(parsed_date=lambda x: pd.to_datetime(x['Date']))
    .groupby('Booking ID')['parsed_date']
    .apply(lambda x: (x.max() - x.min()).days)
)

date_gap_stats = {
    'count_pairs_analyzed': int(len(pairs_df)),
    'mean_day_gap': round(float(pairs_df.mean()), 1),
    'median_day_gap': round(float(pairs_df.median()), 1),
    'min_day_gap': int(pairs_df.min()),
    'max_day_gap': int(pairs_df.max()),
    'pct_same_day': round(float((pairs_df == 0).mean() * 100), 1),
    'pct_within_7_days': round(float((pairs_df <= 7).mean() * 100), 1),
    'pct_over_30_days': round(float((pairs_df > 30).mean() * 100), 1),
    'pct_over_90_days': round(float((pairs_df > 90).mean() * 100), 1),
}

log.add(
    section_id='08_booking_dup_date_gap',
    title='Booking ID Duplicates — Date Gap Between Pairs',
    cell_type='analysis',
    purpose='Measure the date gap between rows sharing a Booking ID to distinguish retry scenarios (same-day) from random ID collision (spread across year)',
    data=date_gap_stats,
)
date_gap_stats

{'count_pairs_analyzed': 1215,
 'mean_day_gap': 120.3,
 'median_day_gap': 105.0,
 'min_day_gap': 0,
 'max_day_gap': 348,
 'pct_same_day': 0.4,
 'pct_within_7_days': 4.6,
 'pct_over_30_days': 82.9,
 'pct_over_90_days': 55.8}

## 8. Booking ID — Numeric Range Analysis

IDs follow format `"CNRxxxxxxx"` (7 digits). If duplicates cluster at specific numeric ranges
it might reveal a generation seeding artifact. If they're uniformly spread, it's random collision.

In [9]:
# Strip surrounding quotes and extract the numeric suffix
df['booking_id_clean'] = df['Booking ID'].str.strip('"')
df['booking_id_num'] = df['booking_id_clean'].str.replace('CNR', '', regex=False).astype(int)

all_id_range = {
    'min_id_num': int(df['booking_id_num'].min()),
    'max_id_num': int(df['booking_id_num'].max()),
    'id_range_span': int(df['booking_id_num'].max() - df['booking_id_num'].min()),
}

dup_ids_numeric = df[dup_booking_mask]['booking_id_num']
dup_id_range = {
    'dup_id_num_min': int(dup_ids_numeric.min()),
    'dup_id_num_max': int(dup_ids_numeric.max()),
    'dup_id_num_mean': round(float(dup_ids_numeric.mean()), 0),
    'dup_id_num_median': round(float(dup_ids_numeric.median()), 0),
}

id_range_result = {**all_id_range, **dup_id_range}

log.add(
    section_id='09_booking_id_numeric_range',
    title='Booking ID Numeric Range',
    cell_type='analysis',
    purpose='Check whether duplicated Booking IDs cluster in a specific numeric range or are uniformly spread, as a signal of generation artifact vs random collision',
    data=id_range_result,
)
id_range_result

{'min_id_num': 1000037,
 'max_id_num': 9999933,
 'id_range_span': 8999896,
 'dup_id_num_min': 1026036,
 'dup_id_num_max': 9987527,
 'dup_id_num_mean': 5547217.0,
 'dup_id_num_median': 5639336.0}

## 9. Customer ID — Repeat Customer vs Malformed Row

If duplicated Customer IDs are repeat customers, they should have different dates/times and different Booking IDs.
If they are malformed, they might share identical rows or show anomalous patterns.

In [10]:
customer_id_counts = df['Customer ID'].value_counts()
dup_customer_mask = df['Customer ID'].duplicated(keep=False)

cid_summary = {
    'total_rows': total_rows,
    'unique_customer_ids': int(df['Customer ID'].nunique()),
    'customer_ids_appearing_more_than_once': int((customer_id_counts > 1).sum()),
    'rows_involved_in_duplicate_customer_ids': int(dup_customer_mask.sum()),
    'pct_rows_affected': round(dup_customer_mask.mean() * 100, 2),
}

# Rides per customer distribution
rides_per_customer = customer_id_counts.value_counts().sort_index()
cid_summary['rides_per_customer_distribution'] = {
    f'customers_with_{k}_rides': int(v) for k, v in rides_per_customer.items()
}

log.add(
    section_id='10_customer_id_counts',
    title='Customer ID Duplicate Counts and Rides Per Customer',
    cell_type='analysis',
    purpose='Quantify Customer ID duplication and determine how many rides per customer, to distinguish repeat customers from malformed IDs',
    data=cid_summary,
)
cid_summary

{'total_rows': 150000,
 'unique_customer_ids': 148788,
 'customer_ids_appearing_more_than_once': 1206,
 'rows_involved_in_duplicate_customer_ids': 2418,
 'pct_rows_affected': np.float64(1.61),
 'rides_per_customer_distribution': {'customers_with_1_rides': 147582,
  'customers_with_2_rides': 1200,
  'customers_with_3_rides': 6}}

## 10. Customer ID Duplicates — Field Variation

Repeat customers should have different Booking IDs and dates but the same Customer ID.
Check whether duplicated Customer IDs share different Booking IDs (= repeat customer) or the same Booking ID (= malformed row).

In [11]:
dup_cid_rows = df[dup_customer_mask].copy()
cid_grouped = dup_cid_rows.groupby('Customer ID')[['Booking ID', 'Date', 'Time', 'Booking Status']].nunique()

# How many groups have >1 unique Booking ID (= repeat customers with different rides)
cid_varies = (cid_grouped > 1).sum()
total_cid_groups = len(cid_grouped)

cid_field_variation = {
    'total_customer_ids_with_duplicates': total_cid_groups,
    'groups_where_booking_id_differs': int(cid_varies['Booking ID']),
    'groups_where_date_differs': int(cid_varies['Date']),
    'groups_where_time_differs': int(cid_varies['Time']),
    'groups_where_status_differs': int(cid_varies['Booking Status']),
    'pct_with_different_booking_ids': round(int(cid_varies['Booking ID']) / total_cid_groups * 100, 1),
}

log.add(
    section_id='11_customer_id_field_variation',
    title='Customer ID Duplicates — Field Variation',
    cell_type='analysis',
    purpose='Determine whether Customer ID duplicates have different Booking IDs and dates (repeat customers) or identical rows (malformed data)',
    data=cid_field_variation,
)
cid_field_variation

{'total_customer_ids_with_duplicates': 1206,
 'groups_where_booking_id_differs': 1206,
 'groups_where_date_differs': 1201,
 'groups_where_time_differs': 1206,
 'groups_where_status_differs': 700,
 'pct_with_different_booking_ids': 100.0}

## 11. Customer ID — Booking Status Mix for Repeat Customers

If repeat customers are real, their rides should have realistic status mixes (mostly Completed, some cancelled etc.).
If all repeat-customer rows are identical in status, it flags a synthetic pattern.

In [12]:
repeat_cid_status = dup_cid_rows['Booking Status'].value_counts()
repeat_cid_status_pct = (repeat_cid_status / len(dup_cid_rows) * 100).round(1)

# Compare with overall status distribution
overall_status = df['Booking Status'].value_counts()
overall_status_pct = (overall_status / len(df) * 100).round(1)

status_comparison = {
    'repeat_customer_rows': {
        'count': repeat_cid_status.to_dict(),
        'pct': repeat_cid_status_pct.to_dict(),
    },
    'overall_dataset': {
        'count': overall_status.to_dict(),
        'pct': overall_status_pct.to_dict(),
    },
}

log.add(
    section_id='12_repeat_customer_status_mix',
    title='Repeat Customer Booking Status Distribution vs Overall',
    cell_type='analysis',
    purpose='Compare booking status distribution for repeat-Customer-ID rows against the overall dataset to check if repeat customers look realistic',
    data=status_comparison,
)
status_comparison

{'repeat_customer_rows': {'count': {'Completed': 1484,
   'Cancelled by Driver': 445,
   'Cancelled by Customer': 175,
   'No Driver Found': 170,
   'Incomplete': 144},
  'pct': {'Completed': 61.4,
   'Cancelled by Driver': 18.4,
   'Cancelled by Customer': 7.2,
   'No Driver Found': 7.0,
   'Incomplete': 6.0}},
 'overall_dataset': {'count': {'Completed': 93000,
   'Cancelled by Driver': 27000,
   'No Driver Found': 10500,
   'Cancelled by Customer': 10500,
   'Incomplete': 9000},
  'pct': {'Completed': 62.0,
   'Cancelled by Driver': 18.0,
   'No Driver Found': 7.0,
   'Cancelled by Customer': 7.0,
   'Incomplete': 6.0}}}

## 12. Sample Duplicate Booking ID Rows (Extended)

Manual inspection of 20 duplicated Booking ID pairs for qualitative pattern recognition.

In [13]:
inspect_cols = ['Booking ID', 'Date', 'Time', 'Customer ID', 'Booking Status', 'Vehicle Type', 'Pickup Location', 'Drop Location']
sample_dup_booking = (
    dup_booking_rows[dup_booking_rows['Booking ID'].isin(size2_ids)]
    .sort_values('Booking ID')[inspect_cols]
    .head(20)
)

log.add(
    section_id='13_sample_dup_booking_rows',
    title='Sample Duplicated Booking ID Rows (Extended)',
    cell_type='analysis',
    purpose='Qualitative inspection of duplicated Booking ID pairs across key fields to identify patterns',
    data=sample_dup_booking.to_dict(orient='records'),
)
sample_dup_booking

,Booking ID,Date,Time,Customer ID,Booking Status,Vehicle Type,Pickup Location,Drop Location
81334,"""CNR1026036""",2024-10-15,18:17:23,"""CID6480133""",Completed,Go Mini,Khandsa,Ashok Vihar
9192,"""CNR1026036""",2024-07-21,17:59:41,"""CID6974869""",No Driver Found,Go Mini,Seelampur,Nehru Place
1353,"""CNR1029172""",2024-01-19,17:00:57,"""CID2615731""",Incomplete,Bike,Jhilmil,Narsinghpur
9587,"""CNR1029172""",2024-12-17,19:19:02,"""CID6382731""",Completed,Auto,Inderlok,Laxmi Nagar
82029,"""CNR1051228""",2024-11-05,16:50:20,"""CID3177617""",Completed,Auto,Dwarka Sector 21,Nehru Place
110412,"""CNR1051228""",2024-01-31,08:42:04,"""CID5242056""",Completed,Premier Sedan,Arjangarh,Yamuna Bank
87333,"""CNR1056023""",2024-04-20,23:16:21,"""CID9367665""",Completed,Premier Sedan,Badshahpur,Sultanpur
120008,"""CNR1056023""",2024-07-07,17:18:49,"""CID4890470""",Incomplete,Go Sedan,Meerut,Tilak Nagar
71570,"""CNR1058956""",2024-02-12,19:02:47,"""CID2451799""",Cancelled by Customer,Premier Sedan,Patel Chowk,Kaushambi
38548,"""CNR1058956""",2024-10-15,13:33:55,"""CID1882606""",Completed,Go Mini,Badshahpur,MG Road


## 13. Sample Repeat Customer ID Rows

Manual inspection of repeat Customer IDs to confirm they look like real repeat bookings.

In [14]:
sample_dup_cid = (
    dup_cid_rows[['Customer ID', 'Booking ID', 'Date', 'Booking Status', 'Vehicle Type', 'Pickup Location']]
    .sort_values('Customer ID')
    .head(20)
)

log.add(
    section_id='14_sample_dup_customer_rows',
    title='Sample Repeat Customer ID Rows',
    cell_type='analysis',
    purpose='Qualitative inspection of rows sharing a Customer ID to confirm repeat customers vs malformed data',
    data=sample_dup_cid.to_dict(orient='records'),
)
sample_dup_cid

,Customer ID,Booking ID,Date,Booking Status,Vehicle Type,Pickup Location
87396,"""CID1008198""","""CNR6195473""",2024-03-28,Cancelled by Driver,Premier Sedan,Aya Nagar
74564,"""CID1008198""","""CNR9208385""",2024-03-18,Completed,Auto,Noida Sector 62
129004,"""CID1008784""","""CNR9673455""",2024-03-02,Completed,Premier Sedan,RK Puram
92529,"""CID1008784""","""CNR4575076""",2024-06-03,Completed,Bike,RK Puram
22845,"""CID1031312""","""CNR7339507""",2024-12-19,Completed,Uber XL,Aya Nagar
90495,"""CID1031312""","""CNR8117971""",2024-12-12,Completed,Auto,Paschim Vihar
146720,"""CID1031683""","""CNR8977869""",2024-05-05,Completed,Bike,India Gate
17661,"""CID1031683""","""CNR6300359""",2024-05-30,No Driver Found,Go Mini,Netaji Subhash Place
350,"""CID1032196""","""CNR1580852""",2024-02-04,Completed,Bike,Mundka
27691,"""CID1032196""","""CNR5491824""",2024-11-27,Incomplete,Go Mini,Vaishali


---
## Save Output

In [15]:
log.save(OUTPUT_DIR / 'id_quality_analysis.json')

Saved → outputs/id_quality_analysis.json  (14 sections)
